<a href="https://colab.research.google.com/github/KateMajzel/ATLAS/blob/main/Tokenizer3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# BYTE-LEVEL BPE — wersja finalna
# baza: 256 bajtów (ID 0-255), merge od ID 256 w górę
# korpus: "Tajemniczy ogród" +"200 mil podwodnej zeglugi" i troche Konstytucji(~9,5 mln znaków)
# ============================================================
import json, time
from collections import Counter

# ===== 1. WCZYTANIE =====
with open("tajemniczy ogrod i 200 mil i kon.txt", encoding="utf-8") as f:
    korpus_tekst = f.read()
print("Korpus surowy:", len(korpus_tekst), "znaków")

# ===== 2. USUNIĘCIE BOILERPLATE (powtarzających się linii) =====
linie = korpus_tekst.split("\n")           # tekst -> lista linii

licznik_linii = {}                          # kreski przy liniach (jak przy parach!)
for linia in linie:
    licznik_linii[linia] = licznik_linii.get(linia, 0) + 1

PROG = 30                                   # linia powtórzona >=30 razy = stopka
czyste_linie = []
for linia in linie:
    if licznik_linii[linia] < PROG:
        czyste_linie.append(linia)

korpus_tekst = "\n".join(czyste_linie)      # sklejam enterami — \n ZOSTAJE
print("Po usunięciu boilerplate:", len(korpus_tekst), "znaków")

# ===== 3. NORMALIZACJA (bez ruszania \n!) =====
korpus_tekst = korpus_tekst.replace("„", '"').replace("”", '"')
korpus_tekst = korpus_tekst.replace("–", "-").replace("—", "-")
while "  " in korpus_tekst:                 # podwójne SPACJE (enterów nie dotyka)
    korpus_tekst = korpus_tekst.replace("  ", " ")
while "\n\n\n" in korpus_tekst:             # potrójne entery -> podwójne (akapit zostaje)
    korpus_tekst = korpus_tekst.replace("\n\n\n", "\n\n")
print("Po normalizacji:", len(korpus_tekst), "znaków")

# ===== 4. TRENING BYTE-LEVEL BPE (4.1 + 4.2 + 4.3) =====
tokeny = list(korpus_tekst.encode("utf-8"))    # start = surowe bajty UTF-8
reguly = {}                                     # (a, b) -> nowy numer ID
next_id = 256                                   # 0-255 zajęte przez bajty
dlugosci = {i: 1 for i in range(256)}           # długość każdego tokenu w bajtach

LICZBA_RUND = 6500
MAX_DL_TOKENU = 20                              # higiena: token max 20 bajtów

print("Tokenów na starcie (bajty):", len(tokeny))
start = time.time()

for runda in range(LICZBA_RUND):
    # 4.1 — zliczanie par (Counter = te same kreski, tylko szybko)
    counts = Counter(zip(tokeny, tokeny[1:]))

    if not counts:                              # bezpiecznik: brak par = koniec
        print("Pary się skończyły w rundzie", runda, "- przerywam.")
        break

    # zwycięzca: najczęstsza para, która nie przekroczy limitu długości
    zwyciezca, najlepszy = None, -1
    for para, ile in counts.items():
        if dlugosci[para[0]] + dlugosci[para[1]] > MAX_DL_TOKENU:
            continue
        if ile > najlepszy:
            najlepszy, zwyciezca = ile, para
    if zwyciezca is None:
        print("Brak dozwolonych par w rundzie", runda, "- przerywam.")
        break

    # 4.2 — merge (przeskok o 2 po parze, o 1 po zwykłym tokenie)
    a, b = zwyciezca
    wynik, i, n = [], 0, len(tokeny)
    while i < n:
        if i < n - 1 and tokeny[i] == a and tokeny[i+1] == b:
            wynik.append(next_id)
            i += 2
        else:
            wynik.append(tokeny[i])
            i += 1
    tokeny = wynik

    reguly[zwyciezca] = next_id
    dlugosci[next_id] = dlugosci[a] + dlugosci[b]
    next_id += 1

    # monitor: czy ostatnie merge jeszcze coś znaczą? (dowód do raportu)
    if runda % 500 == 0:
        print(f"  runda {runda:5d} | zwycięska para: {najlepszy:6d} wystąpień "
              f"| korpus {len(tokeny)} tokenów | {round(time.time()-start)}s")

print("Trening zakończony. Reguł:", len(reguly), "| VOCAB:", 256 + len(reguly))
print("Ostatnia zwycięska para wystąpiła", najlepszy, "razy (jeśli <50 — vocab za duży dla korpusu)")

# ===== 5. VOCAB (slajd 14): numer -> bajty =====
vocab = {i: bytes([i]) for i in range(256)}
for (a, b), new_id in reguly.items():
    vocab[new_id] = vocab[a] + vocab[b]

# ===== 6. ENCODER (4.4): reguły PO KOLEI =====
def encode(tekst, reguly):
    tokeny_nowe = list(tekst.encode("utf-8"))   # ten sam start co w treningu
    for (a, b), new_id in reguly.items():
        wynik, i, n = [], 0, len(tokeny_nowe)
        while i < n:
            if i < n - 1 and tokeny_nowe[i] == a and tokeny_nowe[i+1] == b:
                wynik.append(new_id)
                i += 2
            else:
                wynik.append(tokeny_nowe[i])
                i += 1
        tokeny_nowe = wynik
    return tokeny_nowe

# demonstracja na krótkim zdaniu (UWAGA: w domenie korpusu — nie do metryk!)
nowy_tekst = "Zadzwoniłam do serwisu, bo laptop przestał się ładować, a kurier z nową baterią przyjedzie dopiero w czwartek po południu."
tokeny_id = encode(nowy_tekst, reguly)
tokeny_czytelne = [vocab[t].decode("utf-8", errors="replace") for t in tokeny_id]

print("\nTokeny (czytelne):", tokeny_czytelne)
print("Zdanie demo:", len(nowy_tekst), "znaków |", len(tokeny_id), "tokenów")

# ===== 7. EWALUACJA NA HELD-OUT (uczciwy pomiar — te liczby idą do raportu) =====
with open("heldout.txt", encoding="utf-8") as f:
    heldout = f.read()

ho_ids = encode(heldout, reguly)
ho_znaki = len(heldout)
ho_slowa = len(heldout.split())
ho_tok = len(ho_ids)

metryki = {
    "held_out_znaki": ho_znaki,
    "held_out_slowa": ho_slowa,
    "held_out_tokeny": ho_tok,
    "vocab": 256 + len(reguly),
    "tokens_per_word": round(ho_tok / ho_slowa, 3),
    "tokens_per_1000_chars": round(ho_tok / ho_znaki * 1000, 1),
    "chars_per_token": round(ho_znaki / ho_tok, 3)
}

print("\n=== METRYKI NA HELD-OUT ===")
for k, v in metryki.items():
    print(f"  {k}: {v}")

# ===== 8. JSON — artefakt do oddania =====
wyniki = {
    "dataset": "Tajemniczy ogrod (F.H. Burnett), 200 mil i trochę Konstytucji",
    "typ_tokenizera": "byte-level BPE (baza 256 bajtow, merge od ID 256)",
    "czyszczenie": f"usuniete linie powtarzajace sie >= {PROG} razy (stopki PDF)",
    "limit_dlugosci_tokenu_bajty": MAX_DL_TOKENU,
    "liczba_znakow_korpusu": len(korpus_tekst),
    "liczba_regul_merge": len(reguly),
    "vocab_size": 256 + len(reguly),
    "czestosc_ostatniego_merge": najlepszy,
    "metryki_held_out": metryki,               # <-- Vocab, Tokens/word, Tokens/1000 chars, Chars/token
    "przykladowy_tekst": nowy_tekst,
    "tokeny_id": tokeny_id,
    "tokeny_czytelne": tokeny_czytelne,
    "reguly_merge": [[a, b, new_id] for (a, b), new_id in reguly.items()],
    "vocab": {str(i): vocab[i].decode("utf-8", errors="replace") for i in sorted(vocab)}
}
with open("wyniki.json", "w", encoding="utf-8") as f:
    json.dump(wyniki, f, ensure_ascii=False, indent=2)
print("\nZapisano wyniki.json — pobierz z panelu 📁")

Korpus surowy: 9485253 znaków
Po usunięciu boilerplate: 9363020 znaków
Po normalizacji: 9363020 znaków
Tokenów na starcie (bajty): 10038082
  runda     0 | zwycięska para: 228141 wystąpień | korpus 9809941 tokenów | 4s
  runda   500 | zwycięska para:   1689 wystąpień | korpus 4425087 tokenów | 1630s
  runda  1000 | zwycięska para:    778 wystąpień | korpus 3858658 tokenów | 3092s
  runda  1500 | zwycięska para:    468 wystąpień | korpus 3561193 tokenów | 4515s
  runda  2000 | zwycięska para:    327 wystąpień | korpus 3365717 tokenów | 5923s
  runda  2500 | zwycięska para:    250 wystąpień | korpus 3223363 tokenów | 7296s
  runda  3000 | zwycięska para:    202 wystąpień | korpus 3110955 tokenów | 8663s
  runda  3500 | zwycięska para:    168 wystąpień | korpus 3018847 tokenów | 10001s
  runda  4000 | zwycięska para:    142 wystąpień | korpus 2941759 tokenów | 11354s
  runda  4500 | zwycięska para:    123 wystąpień | korpus 2875733 tokenów | 12722s
  runda  5000 | zwycięska para:    108 w

In [2]:
import subprocess

try:
    # Run gh auth status and capture output
    result = subprocess.run(['gh', 'auth', 'status'], capture_output=True, text=True, check=True)
    print(result.stdout)
    print(result.stderr)
    print("You are authenticated with GitHub CLI.")
except subprocess.CalledProcessError as e:
    print(f"GitHub CLI authentication check failed:\n{e.stderr}")
    print("Please run `gh auth login` in your terminal to authenticate.")
except FileNotFoundError:
    print("GitHub CLI is not installed or not in your PATH. Please install it first.")


GitHub CLI authentication check failed:
You are not logged into any GitHub hosts. To log in, run: gh auth login

Please run `gh auth login` in your terminal to authenticate.
